# InterUni Datathon 2026 - Final Submission Notebook

This notebook records the final modelling approach for **PURESIGMA**.

**Authors**

- Nick Liang
- Zhao Zhang
- Harshvir Singh

The competition objective is binary **log loss**, so lower scores are better. This notebook is intentionally lightweight: it documents the final pipeline, reads saved artifacts, validates the final submission file, and avoids launching any Optuna searches or model training runs.

## 1. Notebook Scope

The experimental work lives in the modelling notebooks and helper scripts. This notebook is the clean handoff version:

- load the train, test, sample submission, and saved model artifacts
- show how EDA and correlation analysis motivated the feature-engineering pass
- summarize the initial modelling path and why the final model became an ensemble
- validate the final submitted CSV
- keep code cells small, named, and easy to audit

Run the **imports** and **paths/helpers** cells below before any other section.

In [1]:
# --- Imports (sections 1–5) ---
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, normalized_mutual_info_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Notebook display defaults — keep tables readable without scrolling forever
pd.set_option("display.max_columns", 80)
pd.set_option("display.precision", 6)

In [2]:
# --- Paths and column names used throughout the notebook ---
ROOT = Path.cwd()

ID_COL = "client_id"
TARGET_COL = "default"
PREDICTION_COL = "default_probability"

TRAIN_PATH = ROOT / "train.csv"
TEST_PATH = ROOT / "test.csv"
SAMPLE_SUBMISSION_PATH = ROOT / "sample_submission.csv"
FINAL_SUBMISSION_PATH = ROOT / "submission_global_targeted_blend.csv"

TARGETED_CONFIG_PATH = ROOT / "global_search_targeted_best.json"
PREVIOUS_GLOBAL_CONFIG_PATH = ROOT / "global_search_best.json"

# Modelling defaults (section 5)
RANDOM_STATE = 42
CV_FOLDS = 5

In [3]:
# --- Small helpers for loading artifacts and sanity-checking submissions ---

def require_file(path: Path) -> Path:
    """Fail fast if a required file is missing — easier to debug than a pandas error later."""
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return path


def load_json(path: Path) -> dict:
    with require_file(path).open(encoding="utf-8") as file:
        return json.load(file)


def read_submission(path: Path) -> pd.DataFrame:
    submission = pd.read_csv(require_file(path))
    required_columns = [ID_COL, PREDICTION_COL]
    missing_columns = [col for col in required_columns if col not in submission.columns]
    if missing_columns:
        raise ValueError(f"{path.name} is missing columns: {missing_columns}")
    return submission[required_columns].copy()


def probability_summary(name: str, values: pd.Series | np.ndarray) -> dict:
    """Quick distribution summary for predicted probabilities."""
    probs = pd.Series(values, dtype="float64")
    return {
        "name": name,
        "rows": int(probs.size),
        "mean": float(probs.mean()),
        "std": float(probs.std(ddof=0)),
        "min": float(probs.min()),
        "p05": float(probs.quantile(0.05)),
        "median": float(probs.quantile(0.50)),
        "p95": float(probs.quantile(0.95)),
        "max": float(probs.max()),
    }


def validate_submission(submission: pd.DataFrame, sample_submission: pd.DataFrame) -> dict:
    """Checks we care about before uploading a CSV to the competition."""
    return {
        "rows_match_sample": bool(len(submission) == len(sample_submission)),
        "ids_match_sample": bool(submission[ID_COL].equals(sample_submission[ID_COL])),
        "has_missing_predictions": bool(submission[PREDICTION_COL].isna().any()),
        "all_probabilities_in_bounds": bool(submission[PREDICTION_COL].between(0.0, 1.0).all()),
        "duplicate_client_ids": int(submission[ID_COL].duplicated().sum()),
    }

## 2. EDA And Data Checks

The `clean.ipynb` pass established the basic schema before modelling. The data uses one row per client, a binary `default` target in train, and a sample-submission file that defines the required test ordering.

The key cleaning outcome was deliberately simple: no missing values were present, the provided numeric encodings were already usable for tree models, and `client_id` was treated only as an identifier.

In [4]:
# Load raw competition files and confirm the basics before any modelling
train_df = pd.read_csv(require_file(TRAIN_PATH))
test_df = pd.read_csv(require_file(TEST_PATH))
sample_submission = pd.read_csv(require_file(SAMPLE_SUBMISSION_PATH))

data_summary = pd.DataFrame(
    [
        {
            "dataset": "train",
            "rows": len(train_df),
            "columns": train_df.shape[1],
            "missing_cells": int(train_df.isna().sum().sum()),
            "duplicate_client_ids": int(train_df[ID_COL].duplicated().sum()),
        },
        {
            "dataset": "test",
            "rows": len(test_df),
            "columns": test_df.shape[1],
            "missing_cells": int(test_df.isna().sum().sum()),
            "duplicate_client_ids": int(test_df[ID_COL].duplicated().sum()),
        },
    ]
)

# Constant-prior baseline — any model must beat predicting the training default rate every time
target_rate = float(train_df[TARGET_COL].mean())
naive_log_loss = float(log_loss(train_df[TARGET_COL], np.repeat(target_rate, len(train_df))))

target_summary = pd.DataFrame(
    [
        {
            "target": TARGET_COL,
            "positive_rate": target_rate,
            "positive_count": int(train_df[TARGET_COL].sum()),
            "negative_count": int((1 - train_df[TARGET_COL]).sum()),
            "constant_rate_log_loss": naive_log_loss,
        }
    ]
)

display(data_summary)
display(target_summary)

,dataset,rows,columns,missing_cells,duplicate_client_ids
0,train,24000,25,0,0
1,test,6000,24,0,0


,target,positive_rate,positive_count,negative_count,constant_rate_log_loss
0,default,0.221208,5309,18691,0.528433


## 3. Correlation Analysis

Before creating new features, we inspected how the original numeric columns related to `default`. The aim was not to pick the final feature set directly, but to identify which parts of the credit history deserved the most feature-engineering attention.

Two complementary measures were used:

- **Spearman correlation** for monotonic relationships with default risk
- **normalized mutual information** for non-linear dependence after binning continuous variables

The raw-column analysis highlighted the repayment-status variables first, especially recent `PAY_*` columns. Payment amounts, bill amounts, and credit limit appeared as secondary signals, so the feature-engineering pass focused on making those histories more explicit.

In [5]:
# Rank raw columns by monotonic (Spearman) and non-linear (NMI) association with default

def discretize_for_nmi(values: pd.Series, n_bins: int = 10) -> pd.Series:
    """Bin continuous values so NMI can pick up non-linear patterns."""
    clean_values = values.replace([np.inf, -np.inf], np.nan)
    fill_value = clean_values.median()
    if pd.isna(fill_value):
        fill_value = 0.0
    clean_values = clean_values.fillna(fill_value)

    if clean_values.nunique(dropna=False) <= n_bins:
        return clean_values.astype(str)

    ranked_values = clean_values.rank(method="first")
    return pd.qcut(ranked_values, q=n_bins, duplicates="drop", labels=False).astype(str)


def correlation_feature_ranking(df: pd.DataFrame, target_col: str, top_n: int = 15) -> pd.DataFrame:
    numeric_cols = [
        col for col in df.select_dtypes(include="number").columns if col != target_col
    ]
    numeric_df = df[numeric_cols + [target_col]].replace([np.inf, -np.inf], np.nan)

    spearman = numeric_df.corr(method="spearman")[target_col].drop(target_col).abs()
    nmi_scores = pd.Series(
        {
            col: normalized_mutual_info_score(
                discretize_for_nmi(numeric_df[col]),
                df[target_col].astype(str),
            )
            for col in numeric_cols
        },
        name="nmi_with_default",
    )

    ranking = pd.concat(
        [spearman.rename("abs_spearman_with_default"), nmi_scores],
        axis=1,
    )
    ranking["mean_rank"] = ranking.rank(ascending=False).mean(axis=1)
    return ranking.sort_values("mean_rank").head(top_n)


raw_correlation_ranking = correlation_feature_ranking(train_df, TARGET_COL)
raw_correlation_ranking

,abs_spearman_with_default,nmi_with_default,mean_rank
PAY_0,0.294178,0.050355,1.0
PAY_2,0.217069,0.028158,3.0
PAY_3,0.198490,0.022206,4.0
PAY_5,0.164239,0.035626,4.0
PAY_4,0.175762,0.019504,5.0
LIMIT_BAL,0.167026,0.009996,6.0
PAY_6,0.146258,0.030720,6.0
PAY_AMT1,0.153009,0.009074,7.5
PAY_AMT2,0.148619,0.008596,8.5
PAY_AMT3,0.132164,0.006807,10.0


## 4. Feature Engineering From Correlation Signals

After the correlation analysis, feature engineering concentrated on the strongest raw signal families:

- repayment-status history from the `PAY_*` columns
- recent and severe delinquency behavior
- payment volume from `PAY_AMT*`
- bill movement from `BILL_AMT*`
- credit exposure through `LIMIT_BAL` and utilization ratios

The first-pass engineered features below came from `clean.ipynb`. They convert month-by-month history into model-friendly summaries such as total payments, total bills, credit utilization, bill trend, delay counts, severe-delay counts, and average repayment status.



| Variable | Data type | Calculation | Brief description |
|---|---|---|---|
| `total_pay` | Numeric / continuous | `PAY_AMT1 + ... + PAY_AMT6` | Total amount paid across the six observed months. |
| `total_bill` | Numeric / continuous | `BILL_AMT1 + ... + BILL_AMT6` | Total billed balance across the six observed months. |
| `credit_util_1` ... `credit_util_6` | Numeric / continuous | `BILL_AMTi / LIMIT_BAL` | Monthly credit utilisation relative to the customer's credit limit. |
| `bill_slope` | Numeric / continuous | Least-squares slope of `BILL_AMT1` ... `BILL_AMT6` over months `1,...,6` | Summarises the overall trend in bill balances across the six months. |
| `bill_abs_change_1_2` ... `bill_abs_change_5_6` | Numeric / continuous | `BILL_AMT(i+1) - BILL_AMTi` | Absolute month-to-month change in bill balance. |
| `bill_pct_change_1_2` ... `bill_pct_change_5_6` | Numeric / continuous | `(BILL_AMT(i+1) - BILL_AMTi) / BILL_AMTi` | Relative month-to-month change in bill balance. Zero denominators are replaced with `NaN`. |
| `bill_abs_change_1_6` | Numeric / continuous | `BILL_AMT6 - BILL_AMT1` | Absolute change in bill balance between months 1 and 6. |
| `bill_pct_change_1_6` | Numeric / continuous | `(BILL_AMT6 - BILL_AMT1) / BILL_AMT1` | Relative change in bill balance between months 1 and 6. |
| `max_delay` | Numeric / ordinal | `max(PAY_0, PAY_2, ..., PAY_6)` | Worst repayment-status value observed across the six months. |
| `num_months_delayed` | Integer / count | `sum(PAY_t > 0)` | Number of months in which the customer had a positive repayment delay. |
| `num_severe_delays` | Integer / count | `sum(PAY_t >= 2)` | Number of months in which the customer was at least two months behind on repayment. |
| `ever_delayed` | Binary integer | `1` if any `PAY_t > 0`, otherwise `0` | Indicates whether the customer experienced any repayment delay during the observed period. |
| `mean_pay_status` | Numeric / ordinal summary | `mean(PAY_0, PAY_2, ..., PAY_6)` | Average repayment-status value across the six months, summarising overall delinquency behaviour. |

In [6]:
# First-pass feature engineering — turn month-by-month history into summary variables
PAY_STATUS_COLS = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
BILL_AMOUNT_COLS = [f"BILL_AMT{i}" for i in range(1, 7)]
PAYMENT_AMOUNT_COLS = [f"PAY_AMT{i}" for i in range(1, 7)]


def make_first_pass_features(df: pd.DataFrame) -> pd.DataFrame:
    """Features from clean.ipynb — same logic we used to guide the later block search."""
    features = df.copy()

    features["total_pay"] = features[PAYMENT_AMOUNT_COLS].sum(axis=1)
    features["total_bill"] = features[BILL_AMOUNT_COLS].sum(axis=1)

    for month, bill_col in enumerate(BILL_AMOUNT_COLS, start=1):
        features[f"credit_util_{month}"] = features[bill_col] / features["LIMIT_BAL"]

    months = np.arange(1, 7)
    month_offsets = months - months.mean()
    bill_values = features[BILL_AMOUNT_COLS].to_numpy()
    features["bill_slope"] = (bill_values * month_offsets).sum(axis=1) / (month_offsets**2).sum()

    for month in range(1, 6):
        current_bill = f"BILL_AMT{month}"
        next_bill = f"BILL_AMT{month + 1}"
        features[f"bill_abs_change_{month}_{month + 1}"] = (
            features[next_bill] - features[current_bill]
        )
        features[f"bill_pct_change_{month}_{month + 1}"] = (
            features[next_bill] - features[current_bill]
        ) / features[current_bill].replace(0, np.nan)

    features["bill_abs_change_1_6"] = features["BILL_AMT6"] - features["BILL_AMT1"]
    features["bill_pct_change_1_6"] = (
        features["BILL_AMT6"] - features["BILL_AMT1"]
    ) / features["BILL_AMT1"].replace(0, np.nan)

    features["max_delay"] = features[PAY_STATUS_COLS].max(axis=1)
    features["num_months_delayed"] = (features[PAY_STATUS_COLS] > 0).sum(axis=1)
    features["num_severe_delays"] = (features[PAY_STATUS_COLS] >= 2).sum(axis=1)
    features["ever_delayed"] = (features[PAY_STATUS_COLS] > 0).any(axis=1).astype(int)
    features["mean_pay_status"] = features[PAY_STATUS_COLS].mean(axis=1)

    return features


analysis_df = make_first_pass_features(train_df)
print(f"Original train shape: {train_df.shape}")
print(f"First-pass feature shape: {analysis_df.shape}")

Original train shape: (24000, 25)
First-pass feature shape: (24000, 51)


In [7]:
# Re-run correlation ranking on engineered features — delay summaries should jump to the top
engineered_correlation_ranking = correlation_feature_ranking(analysis_df, TARGET_COL)
engineered_correlation_ranking

,abs_spearman_with_default,nmi_with_default,mean_rank
num_severe_delays,0.390202,0.094366,1.5
ever_delayed,0.353592,0.102319,2.0
num_months_delayed,0.388466,0.089085,2.5
PAY_0,0.294178,0.050355,4.5
max_delay,0.321376,0.044595,5.0
mean_pay_status,0.258100,0.045747,5.5
PAY_2,0.217069,0.028158,8.0
PAY_3,0.198490,0.022206,9.0
PAY_5,0.164239,0.035626,9.5
PAY_4,0.175762,0.019504,10.0


In [8]:
# Map correlation findings to the feature blocks we tested in later Optuna runs
feature_focus = pd.DataFrame(
    [
        {
            "correlation signal": "Delay severity and frequency",
            "example features": "num_severe_delays, num_months_delayed, ever_delayed, max_delay",
            "model feature blocks": "delay_engineered, pay_status, delay_trends",
            "reason for focus": "These were the strongest monotonic signals after the first feature pass.",
        },
        {
            "correlation signal": "Most recent repayment status",
            "example features": "PAY_0, PAY_2, PAY_3, mean_pay_status",
            "model feature blocks": "pay_status, delay_trends",
            "reason for focus": "Recent delinquency carried more signal than older months.",
        },
        {
            "correlation signal": "Credit exposure and repayment capacity",
            "example features": "LIMIT_BAL, total_pay, PAY_AMT1-6",
            "model feature blocks": "demographics, pay_amounts, models_copy_new_engineered",
            "reason for focus": "Credit limit and repayment volume helped separate risk levels beyond delay counts.",
        },
        {
            "correlation signal": "Bill trajectory and utilization",
            "example features": "total_bill, bill_slope, credit_util_1-6",
            "model feature blocks": "bill_amounts, bill_trends, credit_util, util_stats",
            "reason for focus": "Balance movement and utilization gave secondary credit-risk structure.",
        },
    ]
)

feature_focus

,correlation signal,example features,model feature blocks,reason for focus
0,Delay severity and frequency,"num_severe_delays, num_months_delayed, ever_de...","delay_engineered, pay_status, delay_trends",These were the strongest monotonic signals aft...
1,Most recent repayment status,"PAY_0, PAY_2, PAY_3, mean_pay_status","pay_status, delay_trends",Recent delinquency carried more signal than ol...
2,Credit exposure and repayment capacity,"LIMIT_BAL, total_pay, PAY_AMT1-6","demographics, pay_amounts, models_copy_new_eng...",Credit limit and repayment volume helped separ...
3,Bill trajectory and utilization,"total_bill, bill_slope, credit_util_1-6","bill_amounts, bill_trends, credit_util, util_s...",Balance movement and utilization gave secondar...


The final feature-engineering search therefore started with domain-shaped blocks rather than isolated columns. Delay and repayment-status blocks were treated as core candidates, while utilization, bill trends, repayment amounts, and interaction blocks were tested through cross-validation.

The later Optuna runs confirmed the broad direction but also trimmed noise: the best pulled feature-engineering model kept delay trends and utilization statistics, while dropping the delay-utilization interaction block.

## 5. Initial Modelling Path

The first modelling pipeline used stratified 5-fold cross-validation so each fold kept a similar default/non-default balance. The progression was intentionally simple:

- **Constant-prior baseline** established the minimum model had to beat.
- **Logistic regression** checked whether the correlation-led features had a useful linear probability signal.
- **XGBoost** became the first strong model because it could use non-linear thresholds and interactions in repayment history, utilization, bill movement, and payment amounts.

The initial XGBoost result motivated the broader feature-search work. Rather than manually selecting only the highest-correlation columns, features were grouped into blocks and tested with cross-validated log loss. This preserved the EDA-driven story while still letting validation decide whether secondary features and interactions helped.

We also tested selected features based on correlation as well as 


## Results Interpretation

We noted that in both models using all the variables compared to only selected features resulted in lower log loss compared to using all variables - which suggests that there is some useful signal in all variables even though they did not have high correlaiton with the response.

XGBoost had a lower mean val log loss than logistic and a higher ROC AUC value, indicating more accurate probability predictions iwth respect to the outcomes as well as a more accurate probability rankings. 

We moved forward by gradually tuning the model to observe performance

 
 


In [ ]:
# --- Section 5: baseline vs logistic vs XGBoost on two feature sets ---

DROP_COLS = [ID_COL, TARGET_COL]
all_feature_cols = [col for col in analysis_df.columns if col not in DROP_COLS]

# Same presets used in models.ipynb — compare "everything" vs correlation-led subset
FEATURE_SETS = {
    "all_engineered": all_feature_cols,
    "selected_major": [
        "num_severe_delays",
        "num_months_delayed",
        "ever_delayed",
        "max_delay",
        "PAY_0",
        "mean_pay_status",
        "PAY_2",
        "PAY_3",
        "PAY_4",
        "total_pay",
        "LIMIT_BAL",
    ],
}

y = analysis_df[TARGET_COL]


def get_models():
    """Three-model ladder: prior baseline, linear, then tree-based."""
    return {
        "baseline": DummyClassifier(strategy="prior"),
        "logistic_regression": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=1000,
                        C=1.0,
                        solver="lbfgs",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "xgboost": xgb.XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    }


def run_probability_pipeline(X, y, feature_set_name, plot_roc=True):
    """5-fold CV with log loss + AUC; returns OOF predictions for ROC plots."""
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scoring = {"log_loss": "neg_log_loss", "roc_auc": "roc_auc"}

    results = []
    roc_data = {}

    for model_name, model in get_models().items():
        cv_scores = cross_validate(
            model,
            X,
            y,
            cv=cv,
            scoring=scoring,
            return_train_score=True,
            n_jobs=-1,
        )

        val_log_loss_mean = -cv_scores["test_log_loss"].mean()
        val_roc_auc_mean = cv_scores["test_roc_auc"].mean()

        oof_prob = cross_val_predict(
            model,
            X,
            y,
            cv=cv,
            method="predict_proba",
            n_jobs=-1,
        )[:, 1]

        results.append(
            {
                "feature_set": feature_set_name,
                "model": model_name,
                "train_log_loss_mean": -cv_scores["train_log_loss"].mean(),
                "val_log_loss_mean": val_log_loss_mean,
                "val_log_loss_std": cv_scores["test_log_loss"].std(),
                "train_roc_auc_mean": cv_scores["train_roc_auc"].mean(),
                "val_roc_auc_mean": val_roc_auc_mean,
                "val_roc_auc_std": cv_scores["test_roc_auc"].std(),
            }
        )

        fpr, tpr, _ = roc_curve(y, oof_prob)
        roc_data[model_name] = (fpr, tpr, val_roc_auc_mean)

    results_df = pd.DataFrame(results)

    if plot_roc:
        fig, ax = plt.subplots(figsize=(8, 6))
        for model_name, (fpr, tpr, auc_score) in roc_data.items():
            ax.plot(fpr, tpr, label=f"{model_name} (AUC = {auc_score:.3f})")
        ax.plot([0, 1], [0, 1], "k--", label="random")
        ax.set(xlabel="False positive rate", ylabel="True positive rate")
        ax.set_title(f"ROC curves (OOF CV) — {feature_set_name}")
        ax.legend(loc="lower right")
        plt.tight_layout()
        plt.show()

    return results_df


# Run both feature sets — full engineered matrix vs correlation-selected subset
X_all = analysis_df[FEATURE_SETS["all_engineered"]]
results_all = run_probability_pipeline(X_all, y, "all_engineered")
display(results_all.sort_values("val_log_loss_mean"))

X_selected = analysis_df[FEATURE_SETS["selected_major"]]
results_selected = run_probability_pipeline(X_selected, y, "selected_major")
display(results_selected.sort_values("val_log_loss_mean"))

In [ ]:
# XGBoost feature importance — fit on full training data for presentation slides
TOP_N_IMPORTANCE = 15

xgb_model = get_models()["xgboost"]
xgb_model.fit(X_all, y)

importance = pd.DataFrame(
    {
        "feature": X_all.columns,
        "importance": xgb_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

importance["importance_pct"] = 100 * importance["importance"] / importance["importance"].sum()
top_importance = importance.head(TOP_N_IMPORTANCE).iloc[::-1]

display(importance.head(TOP_N_IMPORTANCE))

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(
    data=top_importance,
    x="importance_pct",
    y="feature",
    hue="feature",
    palette="Blues_r",
    legend=False,
    ax=ax,
)
ax.set_xlabel("Gain importance (% of total)")
ax.set_ylabel("")
ax.set_title(f"Top {TOP_N_IMPORTANCE} XGBoost features (all_engineered, full-train fit)")
plt.tight_layout()
plt.show()

## 6. Final Model Architecture

The final submitted model is an ensemble, not a single estimator. That matters because the public leaderboard showed the same pattern as local CV: blends beat individual model families.

The architecture is:

1. **Feature layer** - engineered credit-risk features motivated by the correlation analysis and grouped into reusable feature blocks.
2. **Base model layer** - XGBoost, LightGBM, and CatBoost candidates from saved Optuna runs.
3. **OOF prediction layer** - each candidate is evaluated through stratified folds to produce out-of-fold probabilities.
4. **Meta layer** - a constrained weighted blend is optimized for log loss.
5. **Final output** - selected blend weights are applied to full-train refits to create `submission_global_targeted_blend.csv`.

Calibration and logistic stacking were tested as post-processing alternatives, but the raw constrained blend had the lowest local log loss and was selected.

This section reads only the saved global-search artifacts. It intentionally does not read the live `feature_eng_joint_best.json`, because that file can change during ongoing feature-engineering experiments.

In [10]:
targeted_config = load_json(TARGETED_CONFIG_PATH)
previous_global_config = load_json(PREVIOUS_GLOBAL_CONFIG_PATH) if PREVIOUS_GLOBAL_CONFIG_PATH.exists() else None

result_rows = [
    {
        "artifact": "targeted global blend",
        "source_file": TARGETED_CONFIG_PATH.name,
        "local_cv_log_loss": targeted_config["blend"]["val_log_loss_mean"],
        "local_cv_auc": targeted_config["blend"]["val_roc_auc_mean"],
        "selected": targeted_config.get("selected_submission") == "blend",
    }
]

if targeted_config.get("calibrated_blend"):
    result_rows.append(
        {
            "artifact": "targeted calibrated blend",
            "source_file": TARGETED_CONFIG_PATH.name,
            "local_cv_log_loss": targeted_config["calibrated_blend"]["val_log_loss_mean"],
            "local_cv_auc": targeted_config["calibrated_blend"]["val_roc_auc_mean"],
            "selected": targeted_config.get("selected_submission") == "calibrated_blend",
        }
    )

if previous_global_config:
    result_rows.append(
        {
            "artifact": "previous global blend",
            "source_file": PREVIOUS_GLOBAL_CONFIG_PATH.name,
            "local_cv_log_loss": previous_global_config["blend"]["val_log_loss_mean"],
            "local_cv_auc": previous_global_config["blend"]["val_roc_auc_mean"],
            "selected": False,
        }
    )

model_result_summary = pd.DataFrame(result_rows).sort_values("local_cv_log_loss")
model_result_summary

,artifact,source_file,local_cv_log_loss,local_cv_auc,selected
0,targeted global blend,global_search_targeted_best.json,0.422861,0.790076,True
1,targeted calibrated blend,global_search_targeted_best.json,0.422940,0.789938,False
2,previous global blend,global_search_best.json,0.422985,0.789890,False


In [11]:
blend_weights = (
    pd.Series(targeted_config["blend"]["weights"], name="weight")
    .rename_axis("model_label")
    .reset_index()
    .sort_values("weight", ascending=False)
    .reset_index(drop=True)
)
blend_weights["weight_pct"] = 100.0 * blend_weights["weight"]

print(f"Number of blended candidates: {len(blend_weights)}")
print(f"Weight sum: {blend_weights['weight'].sum():.6f}")
blend_weights.head(15)

Number of blended candidates: 17
Weight sum: 1.000000


,model_label,weight,weight_pct
0,optuna_original_catboost_trial_17_all_engineered,0.193763,19.376328
1,global_search_xgboost_trial_42_exported_featur...,0.188081,18.808146
2,global_search_xgboost_trial_41_exported_featur...,0.186552,18.655245
3,optuna_extended_catboost_trial_28_all_engineer...,0.130339,13.033925
4,global_search_lightgbm_trial_41_all_features,0.077645,7.764533
5,optuna_original_lightgbm_trial_12_all_engineered,0.042849,4.284947
6,global_search_xgboost_trial_43_exported_featur...,0.041193,4.119272
7,global_search_lightgbm_trial_44_all_features,0.039707,3.970748
8,feature_eng_joint_xgboost_trial_saved_feature_...,0.035260,3.525957
9,global_search_lightgbm_trial_37_all_features,0.023435,2.343498


In [ ]:
# check models.ipynb for initial setup

## 7. Public Leaderboard Context

These public scores came from the latest submission dashboard. They are used here only as external feedback on direction, not as a tuning target.

In [12]:
public_scores = pd.DataFrame(
    [
        {"submission": "submission_global_targeted_blend.csv", "public_log_loss": 0.41217},
        {"submission": "submission_global_blend.csv", "public_log_loss": 0.41247},
        {"submission": "submission_feature_eng.csv", "public_log_loss": 0.41271},
        {"submission": "submission_xgboost.csv", "public_log_loss": 0.41280},
        {"submission": "submission_blend.csv", "public_log_loss": 0.41409},
        {"submission": "submission_lightgbm.csv", "public_log_loss": 0.41498},
        {"submission": "submission_catboost.csv", "public_log_loss": 0.41535},
    ]
).sort_values("public_log_loss")

public_scores["delta_vs_best"] = public_scores["public_log_loss"] - public_scores["public_log_loss"].min()
public_scores

,submission,public_log_loss,delta_vs_best
0,submission_global_targeted_blend.csv,0.41217,0.00000
1,submission_global_blend.csv,0.41247,0.00030
2,submission_feature_eng.csv,0.41271,0.00054
3,submission_xgboost.csv,0.41280,0.00063
4,submission_blend.csv,0.41409,0.00192
5,submission_lightgbm.csv,0.41498,0.00281
6,submission_catboost.csv,0.41535,0.00318


## 8. Final Submission Validation

The selected file is `submission_global_targeted_blend.csv`. Before submitting or sharing it, validate row count, ID ordering, duplicates, missing values, and probability bounds.

In [13]:
final_submission = read_submission(FINAL_SUBMISSION_PATH)
validation_report = pd.DataFrame([validate_submission(final_submission, sample_submission)])
probability_report = pd.DataFrame(
    [probability_summary(FINAL_SUBMISSION_PATH.name, final_submission[PREDICTION_COL])]
)

display(validation_report)
display(probability_report)
final_submission.head()

,rows_match_sample,ids_match_sample,has_missing_predictions,all_probabilities_in_bounds,duplicate_client_ids
0,True,True,False,True,0


,name,rows,mean,std,min,p05,median,p95,max
0,submission_global_targeted_blend.csv,6000,0.218828,0.195017,0.028065,0.046959,0.14532,0.691944,0.852472


,client_id,default_probability
0,CC_0012A082B7B7,0.127502
1,CC_0012BC27DFB6,0.128202
2,CC_001563B2143D,0.111690
3,CC_001B46930B6F,0.093322
4,CC_001D771E1C9F,0.043075


In [14]:
comparison_files = [
    ROOT / "submission_global_targeted_blend.csv",
    ROOT / "submission_global_blend.csv",
    ROOT / "submission_feature_eng.csv",
    ROOT / "submission_xgboost.csv",
    ROOT / "submission_lightgbm.csv",
    ROOT / "submission_catboost.csv",
]

submission_summaries = []
for submission_path in comparison_files:
    if submission_path.exists():
        submission = read_submission(submission_path)
        submission_summaries.append(
            probability_summary(submission_path.name, submission[PREDICTION_COL])
        )

pd.DataFrame(submission_summaries).sort_values("name")

,name,rows,mean,std,min,p05,median,p95,max
5,submission_catboost.csv,6000,0.218748,0.194825,0.022347,0.049905,0.141752,0.691550,0.894570
2,submission_feature_eng.csv,6000,0.218639,0.196885,0.024210,0.044548,0.144754,0.696027,0.857077
1,submission_global_blend.csv,6000,0.218777,0.194547,0.027750,0.047897,0.145294,0.690509,0.851441
0,submission_global_targeted_blend.csv,6000,0.218828,0.195017,0.028065,0.046959,0.145320,0.691944,0.852472
4,submission_lightgbm.csv,6000,0.219429,0.195674,0.022277,0.045982,0.145960,0.696575,0.854969
3,submission_xgboost.csv,6000,0.219065,0.195657,0.021306,0.046491,0.144810,0.697403,0.830244


## 9. Reproducibility Notes

Create the local environment:

```bash
/opt/homebrew/bin/python3.13 -m venv .venv
.venv/bin/python -m pip install --upgrade pip
.venv/bin/python -m pip install -r requirements.txt
```

Rebuild the current best search artifact and submission. This is intentionally not executed in this notebook because it starts model training:

```bash
.venv/bin/python -u run_global_search.py   --trials-per-model 50   --models xgboost lightgbm catboost   --search-models xgboost lightgbm   --top-k-per-model 5   --blend-trials 1500   --calibration-trials 800   --disable-stack   --output global_search_targeted_best.json   --submission submission_global_targeted_blend.csv   --make-submission
```

The next architecture improvement should be an OOF prediction cache and model registry, so future blend searches can run without repeatedly refitting every base model.